# Single-Heading Pre-trained CNN

# Imports, Primitives

In [ ]:
from google.colab import drive
import os
import pandas as pd
import os
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import numpy as np
import requests
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, losses, regularizers
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import seaborn as sns

drive.mount('/content/drive')

####### SAATWIK CODE

import os, time

ZIP_SRC  = "/content/drive/MyDrive/Deep Learning Project/GoogleStreetViewImagesNew.zip"
LOCAL_DIR = "/content/streetview_dataset"

if not os.path.exists(LOCAL_DIR):
    t0 = time.time()
    !cp "{ZIP_SRC}" /content/streetview_dataset.zip
    !unzip -q /content/streetview_dataset.zip -d {LOCAL_DIR}
    print(f"Staged in {time.time() - t0:.1f}s")

####### SAATWIK CODE
!ls {LOCAL_DIR} | head

Mounted at /content/drive
Staged in 66.4s
GoogleStreetViewImagesNew
__MACOSX


## Data Loading Function

In [ ]:
# Precursor Functions

###################
# I. Data loading #
###################
# Sources:
# Assignment I dataloading fn &
# https://medium.com/@kumudtraveldiaries/step-by-step-preprocessing-guide-for-images-in-both-cnn-and-dense-layer-pipelines-1994c3ad3e87

def dataloading(csv_path="/content/drive/MyDrive/Deep Learning Project/images_bra-2.csv",
                img_size=224,
                batch_size=32,
                preprocess_fn=None,
                ordinal=None):

    # 1. Load CSV
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=["income_group"]).reset_index(drop=True)

    # 2. Build file paths and group IDs
    # CORRECT — file_path already contains GoogleStreetViewImagesNew/filename
    df["full_path"] = df["file_path"].apply(
        lambda f: os.path.join("/content/streetview_dataset/", f)
    )
    df["loc_id"] = df["file_path"].apply(
        lambda f: os.path.basename(f).split("_h")[0]
    )

    # 3. ── Filter missing files before splitting ──────────────────────────────
    before = len(df)
    df = df[df["full_path"].apply(os.path.exists)].reset_index(drop=True)
    print(f"Dropped {before - len(df)} missing files, {len(df)} remaining")
    # ─────────────────────────────────────────────────────────────────────────

    # 4. Train / Val / Test split at location level
    locs = df["loc_id"].unique()
    np.random.seed(50)
    np.random.shuffle(locs)
    n_test = int(len(locs) * 0.15)
    n_val  = int(len(locs) * 0.15)
    test_locs  = locs[:n_test]
    val_locs   = locs[n_test:n_test + n_val]
    train_locs = locs[n_test + n_val:]

    train_df = df[df["loc_id"].isin(train_locs)]
    val_df   = df[df["loc_id"].isin(val_locs)]
    test_df  = df[df["loc_id"].isin(test_locs)]

    def to_ordinal(y, num_classes=3):
        out = np.zeros((len(y), num_classes - 1), dtype=np.float32)
        for i, label in enumerate(y):
            out[i, :label] = 1.0
        return out

    train_labels = train_df["income_group"].values.astype(np.int64)
    val_labels   = val_df["income_group"].values.astype(np.int64)
    test_labels  = test_df["income_group"].values.astype(np.int64)

    if ordinal:
        train_labels = to_ordinal(train_labels, num_classes=3)
        val_labels   = to_ordinal(val_labels,   num_classes=3)
        test_labels  = to_ordinal(test_labels,  num_classes=3)

    splits = {
        "train": (train_df["full_path"].values, train_labels),
        "val":   (val_df["full_path"].values,   val_labels),
        "test":  (test_df["full_path"].values,  test_labels),
    }

    for k, (p, l) in splits.items():
        print(f"{k}: {len(p)} images")

    # 5. Image loading function
    def load_image(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, [img_size, img_size])
        img = tf.cast(img, tf.float32)
        if preprocess_fn:
            img = preprocess_fn(img)
        else:
            img = img / 255.0
        return img, label

    # 6. Build tf.data pipelines
    def make_dataset(file_paths, lbls, is_train=False):
        ds = tf.data.Dataset.from_tensor_slices((file_paths, lbls))
        ds = ds.cache()
        if is_train:
            ds = ds.shuffle(len(file_paths))
        ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        return ds

    train_ds = make_dataset(*splits["train"], is_train=True)
    val_ds   = make_dataset(*splits["val"])
    test_ds  = make_dataset(*splits["test"])

    return train_ds, val_ds, test_ds

# ConvNeXtBase Backbone Single-Heading Prediction

A pre-trained ConvNeXtBase backbone serves as a frozen feature extractor, with a lightweight custom classification head trained on top.

## Architecture

    Input Image (H × W × C)
            ↓
      Data Augmentation        ← random transforms at training time only
            ↓
    ConvNeXtBase Backbone      ← frozen, pre-trained on ImageNet (~89M params)
            ↓
    GlobalAveragePooling2D     ← collapses (H', W', 1024) → (1024,)
            ↓
      Dense(3, softmax)        ← outputs class probabilities

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os

# ── 1. Load data ───────────────────────────────────────────────────────────────
# ConvNeXt's preprocess_input scales pixels to [-1, 1] internally
train_ds, val_ds, test_ds = dataloading(
    csv_path="/content/drive/MyDrive/Deep Learning Project/images_bra-3.csv",
    img_size=224,           # ConvNeXtBase pretrained at 224 — standard resolution
    batch_size=16,
    preprocess_fn=tf.keras.applications.convnext.preprocess_input
)

IMG_SHAPE = (224, 224, 3)

# ── 2. Verify shapes ───────────────────────────────────────────────────────────
image_batch, label_batch = next(iter(train_ds))
print("Image batch shape :", image_batch.shape)   # (16, 224, 224, 3)
print("Label batch shape :", label_batch.shape)
print("Pixel value range :", image_batch.numpy().min(), "→", image_batch.numpy().max())

# ── 3. Augmentation ────────────────────────────────────────────────────────────
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
], name="data_augmentation")

# ── 4. ConvNeXtBase backbone ───────────────────────────────────────────────────
# Outputs (None, 7, 7, 1024) at 224×224 — uses LayerNorm instead of BatchNorm,
# making it more stable with small batch sizes during fine-tuning
base_model = tf.keras.applications.ConvNeXtBase(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

# ── 5. Build model ─────────────────────────────────────────────────────────────
# GAP → Dropout → Dense(512) → Dense(3) — same head as before
inputs = tf.keras.Input(shape=IMG_SHAPE, name="input_images")
x = data_augmentation(inputs)

# training=False keeps the backbone in inference mode during Phase 1
# ConvNeXt uses LayerNorm (not BatchNorm), so this is less critical than
# EfficientNet — but still recommended for Phase 1 feature extraction
x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
x = tf.keras.layers.Dropout(0.4, name="dropout_1")(x)
x = tf.keras.layers.Dense(
        512, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4),
        name="dense_hidden"
    )(x)
x = tf.keras.layers.Dropout(0.3, name="dropout_2")(x)
outputs = tf.keras.layers.Dense(
            3, activation="softmax",
            kernel_regularizer=tf.keras.regularizers.l2(1e-4),
            name="predictions"
          )(x)

model = tf.keras.Model(inputs, outputs, name="ConvNeXtBase_Improved")
model.summary()

# ── 6. Callbacks ───────────────────────────────────────────────────────────────
def make_callbacks(ckpt="best_convnextbase.keras"):
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=10,
            restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=ckpt, monitor='val_accuracy',
            save_best_only=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.2, patience=4,
            min_lr=1e-6, verbose=1
        ),
    ]

# ── 7. Phase 1 — Feature extraction (backbone frozen) ─────────────────────────
print("\n=== Phase 1: Feature Extraction (frozen backbone) ===")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=make_callbacks(),
    verbose=1
)

print("\n=== Test Evaluation (Phase 1) ===")
loss_p1, acc_p1 = model.evaluate(test_ds, verbose=1)
print(f"Test accuracy : {acc_p1:.4f}  |  Test loss : {loss_p1:.4f}")

# ── 8. Phase 2 — Fine-tuning ───────────────────────────────────────────────────
print("\n=== Phase 2: Fine-Tuning (top layers unfrozen) ===")

base_model.trainable = True

# ConvNeXtBase has ~390 layers; freeze first 270 (~70%), unfreeze top ~30%
# ConvNeXt uses LayerNorm — NO need to re-freeze norm layers (unlike EfficientNet's BN)
# This makes fine-tuning simpler and generally more stable
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Trainable backbone layers: {trainable_count} / {len(base_model.layers)}")

# Cosine decay — well-suited to ConvNeXt's training recipe
total_steps = 30 * len(train_ds)
cosine_lr = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-5,
    decay_steps=total_steps,
    alpha=1e-7,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=cosine_lr),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_convnextbase.keras",
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
]

history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks_p2,
    verbose=1
)

print("\n=== Test Evaluation (Phase 2 — Fine-tuned) ===")
loss_p2, acc_p2 = model.evaluate(test_ds, verbose=1)
print(f"Test accuracy : {acc_p2:.4f}  |  Test loss : {loss_p2:.4f}")

Dropped 0 missing files, 21906 remaining
train: 15338 images
val: 3284 images
test: 3284 images
Image batch shape : (16, 224, 224, 3)
Label batch shape : (16,)
Pixel value range : 0.0 → 255.0


Model: "ConvNeXtBase_Improved"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_images (InputLayer)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ convnext_base (Functional)      │ (None, 7, 7, 1024)     │    87,566,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 88,092,803 (336.05 MB)

 Trainable params: 526,339 (2.01 MB)

 Non-trainable params: 87,566,464 (334.04 MB)


=== Phase 1: Feature Extraction (frozen backbone) ===
Epoch 1/20
959/959 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.4554 - loss: 1.1502
Epoch 1: val_accuracy improved from None to 0.47533, saving model to best_convnextbase.keras

Epoch 1: finished saving model to best_convnextbase.keras
959/959 ━━━━━━━━━━━━━━━━━━━━ 128s 113ms/step - accuracy: 0.4810 - loss: 1.0986 - val_accuracy: 0.4753 - val_loss: 1.0802 - learning_rate: 0.0010
Epoch 2/20
958/959 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.5265 - loss: 1.0412
Epoch 2: val_accuracy improved from 0.47533 to 0.56151, saving model to best_convnextbase.keras

Epoch 2: finished saving model to best_convnextbase.keras
959/959 ━━━━━━━━━━━━━━━━━━━━ 87s 91ms/step - accuracy: 0.5307 - loss: 1.0359 - val_accuracy: 0.5615 - val_loss: 0.9875 - learning_rate: 0.0010
Epoch 3/20
958/959 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.5337 - loss: 1.0235
Epoch 3: val_accuracy did not improve from 0.56151
959/959 ━━━━━━━━━━━━━━━━━━━━ 85s 89ms/